In [22]:
import pandas as pd
import numpy as np
from collections import defaultdict
import requests
import time

In [26]:
traffic_data = pd.read_csv('datasets/unique_locations.csv')
print(traffic_data.head())

                       Location   Longitude   Latitude
0     AUBURN_RD N of BURWOOD_RD  145.045675 -37.823965
1     AUBURN_RD S of BURWOOD_RD  145.043460 -37.825420
2     BALWYN_RD N OF BELMORE_RD  145.081970 -37.804310
3  BALWYN_RD N OF WHITEHORSE_RD  145.080100 -37.814040
4  BALWYN_RD N of CANTERBURY_RD  145.078000 -37.825060


In [31]:
connections = defaultdict(set)

for loc_string in traffic_data['Location']:
    loc_string = loc_string.lower().strip()
    segments = loc_string.split()
    if len(segments) < 2:
        continue  # skip invalid entries

    origin = segments[0]
    target = segments[-1]
    check_segments_origin = segments[1]
    check_segments_last = segments[-2]

    if len(check_segments_origin) > 2:
        # concatenate origin and check_segments
        origin = f"{origin} {check_segments_origin}"

    if check_segments_last != 'of':
        # concatenate target and check_segments_last
        target = f"{check_segments_last} {target}"

    # Add connection both ways (undirected)
    connections[origin].add(target)
    connections[target].add(origin)

# Step 2: Convert to DataFrame
new_df = pd.DataFrame({
    'Location': list(connections.keys()),
    'Connected Nodes': [sorted(list(neighbors)) for neighbors in connections.values()]
})

print(new_df.head())
# Step 3: Save to CSV
new_df.to_csv('datasets/unique_locations_connections.csv', index=False)

        Location                                    Connected Nodes
0      auburn_rd                                       [burwood_rd]
1     burwood_rd    [auburn_rd, bridge_rd, glenferrie_rd, power_st]
2      balwyn_rd  [belmore_rd, canterbury_rd, doncaster_rd, whit...
3     belmore_rd                              [balwyn_rd, burke_rd]
4  whitehorse_rd                              [balwyn_rd, burke_rd]


In [32]:
def get_coordinates_osm(location_name):
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        'q': f"{location_name}, Boroondara, Victoria, Australia",
        'format': 'json',
        'limit': 1
    }
    headers = {'User-Agent': 'location-mapper/1.0'}
    
    response = requests.get(url, params=params, headers=headers)
    if response.status_code == 200 and response.json():
        result = response.json()[0]
        return float(result['lat']), float(result['lon'])
    else:
        return None, None

# Add coordinates to each row
locations = new_df['Location'].tolist()
latitudes = []
longitudes = []

for loc in locations:
    lat, lon = get_coordinates_osm(loc)
    latitudes.append(lat)
    longitudes.append(lon)
    print(f"Fetched: {loc} → ({lat}, {lon})")
    time.sleep(1)  # Respect OSM rate limit

# Add to DataFrame
new_df['Latitude'] = latitudes
new_df['Longitude'] = longitudes

# Save for future use
new_df.to_csv("graph_locations.csv", index=False)

Fetched: auburn_rd → (-37.8164674, 145.0461811)
Fetched: burwood_rd → (-37.822703, 145.0361929)
Fetched: balwyn_rd → (-37.7902706, 145.0857682)
Fetched: belmore_rd → (-37.8034033, 145.102324)
Fetched: whitehorse_rd → (-37.8114601, 145.0698677)
Fetched: canterbury_rd → (-37.8243743, 145.084124)
Fetched: doncaster_rd → (-37.7889745, 145.1019751)
Fetched: barkers_rd → (-37.8125829, 145.0191168)
Fetched: denmark_st → (-37.8078957, 145.0289831)
Fetched: high_st → (-37.8639307, 145.0841172)
Fetched: burke_rd → (-37.7858386, 145.0639673)
Fetched: church_st → (-37.8192467, 145.0173803)
Fetched: bridge_rd → (-37.8199456, 145.0159701)
Fetched: bulleen_rd → (-37.7847078, 145.0776482)
Fetched: thompsons_rd → (-37.7792685, 145.081886)
Fetched: eastern_fwy_w_bd_ramps → (None, None)
Fetched: harp_rd → (-37.7999045, 145.0554133)
Fetched: mont albert_rd → (-37.8168314, 145.0676484)
Fetched: riversdale_rd → (-37.8316157, 145.0589298)
Fetched: toorak_rd → (-37.8452216, 145.0432696)
Fetched: eastern_fwy →

In [ ]:
df_full = pd.read_csv("./datasets/temp.csv")   # contains 'Location' and 'Site Type'
df_partial = pd.read_csv("./datasets/node_id_to_location.csv") # contains 'Location' only

# Clean formatting (optional but recommended)
df_full['Location'] = df_full['Location'].str.upper().str.strip()
df_partial['Location'] = df_partial['Location'].str.upper().str.strip()

# Match using substring logic
matched_site_types = []

for partial_loc in df_partial['Location']:
    match = df_full[df_full['Location'].str.contains(partial_loc, na=False)]
    if not match.empty:
        site_type = match.iloc[0]['Site Type']  # take the first match
    else:
        site_type = None
    matched_site_types.append(site_type)

# Assign to partial dataframe
df_partial['Site Type'] = matched_site_types
df_partial['Location'] = df_partial['Location'].str.lower()  

df_partial.to_csv("csv2_with_site_type_matched.csv", index=False)